# Transform: make a dataset, with no model at all

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fideus-labs/KonfAI/blob/main/examples/Transform/Transform_demo.ipynb)

**Run all the cells.** No network, no checkpoint, no training loop: this notebook folds a
cohort into one template, then draws four augmented copies of every case. About a minute on
CPU, on 3.5 MB of synthetic data generated right here. Nothing is downloaded.


In [ ]:
# Setup: find KonfAI (cloning it on Colab), install what is missing, load the notebook helpers.
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    REPO_DIR = Path("/content/KonfAI")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/fideus-labs/KonfAI", str(REPO_DIR)], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples").is_dir())
sys.path.insert(0, str(REPO_DIR / "examples"))

from konfai_demo import read, run, setup, show

EXAMPLE_DIR, DATASET_DIR, DEVICE = setup(REPO_DIR, "Transform", ("konfai", f"{REPO_DIR}[imaging]"), "matplotlib")


## 1. A cohort that does not share a grid

`make_dataset.py` writes six volumes, each with its own extent, spacing and origin. That is the
ordinary state of a cohort as acquired, and it is what makes the example worth running.


In [ ]:
run(sys.executable, "make_dataset.py")

import SimpleITK as sitk

for case in sorted(p.name for p in (EXAMPLE_DIR / "Raw").iterdir()):
    image = sitk.ReadImage(str(EXAMPLE_DIR / "Raw" / case / "CT.mha"))
    print(f"{case}: shape {image.GetSize()}  spacing {tuple(round(s, 2) for s in image.GetSpacing())}")


## 2. Read the plan before it writes anything

Every TRANSFORM run prints its plan first. `--plan` prints it and stops, so you see how each
case will be routed (`STREAM`, `LOAD`, `WHOLE-VOLUME`, `SKIP`, `REDUCE`) before a byte is
written.


In [ ]:
run("konfai", "TRANSFORM", "--config", "Transform.yml", "--plan")


## 3. N cases, one volume

`Transform.yml` clips each case, puts the cohort on one member's grid, then `Reduce` folds it
at fixed voxel. Everything above the marker runs per case, everything below runs once on the
result. The engine walks the output's regions and the cases within each, so the peak is a few
regions rather than six volumes.


In [ ]:
run("konfai", "TRANSFORM", "-y", "--config", "Transform.yml")

template = read(EXAMPLE_DIR / "Template" / "template" / "CT_template.mha")
first = read(EXAMPLE_DIR / "Raw" / "CASE_000" / "CT.mha")
middle = template.shape[0] // 2
show([("CASE_000", first[first.shape[0] // 2], "gray", (0, 400)),
      ("Median template", template[middle], "gray", (0, 400))])


## 4. One case, N copies

`Transform_expand.yml` is the mirror. `Expand` multiplies, and the stages after it run once per
copy: here a brightness draw. Each draw derives from `manual_seed` and the case, so two chains
asked for the same case produce the same copies, which is how an image and its mask stay paired.


In [ ]:
run("konfai", "TRANSFORM", "-y", "--config", "Transform_expand.yml")

copies = sorted((EXAMPLE_DIR / "Augmented").glob("CASE_000_r*"))
print("copies of CASE_000:", [p.name for p in copies])
show([(p.name, read(p / "CT_aug.mha")[first.shape[0] // 2], "gray", (0, 400)) for p in copies])


## What to change next

- **another fold**: `Reduce.operator` takes `Mean`, `Median`, `Std` for the voxel-wise spread of
  a cohort, `Vote` for label maps, `Concat` to stack them side by side.
- **your own data**: point `dataset_filenames` at a folder of `<case>/CT.mha` and keep the rest.
- **another output**: `Write` takes a format token, so `./Out:omezarr` writes an OME-Zarr store
  and `./Fields:itktransform` writes an ITK transform 3D Slicer opens directly.

`README.md` next to this notebook covers both configs key by key.
